# Train Word2Vec on British fiction

Minimal pipeline for training a Word2Vec model from a folder of raw
`.txt` novels. Three stages:

1. **Clean** — read each novel, split it into sentences (on `.`, `!`, `?`),
   tokenise, lowercase. Write one sentence per line to
   `<folder>_normalized/`.
2. **Train** — run gensim's Word2Vec on the cleaned corpus.
3. **Save** — drop the `.model` file into `<folder>_model/`.

> Gensim itself does NOT clean text — it takes tokens as-is. We do all
> the cleaning ourselves in stage 1, and the cleaned files are kept on
> disk so you can inspect them and re-run training without re-cleaning.

## Setup

Edit `INPUT_DIR` to point at your folder of novels. The other two
folders are derived automatically.

`os.path.expanduser` lets you use `~` in the path on any OS.

In [2]:
import os
from pathlib import Path

# === EDIT ME ===
INPUT_DIR = Path(os.path.expanduser("~/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus"))

# Derived paths — created next to INPUT_DIR.
NORMALIZED_DIR = INPUT_DIR.parent / f"{INPUT_DIR.name}_normalized"
MODEL_DIR      = INPUT_DIR.parent / f"{INPUT_DIR.name}_model"

# Word2Vec hyperparameters — easy to tweak.
W2V_PARAMS = {
    "vector_size": 100,    # embedding dimension
    "window":      5,      # context window
    "min_count":   5,      # ignore words below this frequency
    "sg":          0,      # 0 = CBOW, 1 = skip-gram
    "workers":     4,      # parallelism
    "epochs":      5,      # training passes
    "seed":        42,     # reproducibility
}

print(f"Input:      {INPUT_DIR}")
print(f"Normalized: {NORMALIZED_DIR}")
print(f"Model:      {MODEL_DIR}")

Input:      /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus
Normalized: /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus_normalized
Model:      /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus_model


## Stage 1 — clean

For every `.txt` file in `INPUT_DIR`:

1. Read the whole file.
2. Split into sentences on `.`, `!`, `?`.
3. Tokenise each sentence: keep runs of letters (Latin + extended Latin)
   with optional hyphens/digits inside; lowercase.
4. Drop empty lines.
5. Write to `<file>.txt` in `NORMALIZED_DIR`, one sentence per line.

The cleaned files match what `gensim.models.word2vec.PathLineSentences`
expects: a folder of text files where each line is one sentence.

In [3]:
import re

# Sentence splitter — fires on . ! ? followed by whitespace.
# Edge cases like "Mr." and "U.S.A." may produce one extra sentence
# boundary; for novel-length corpora that's a negligible source of noise.
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

# Token = letters (incl. extended Latin / diacritics), optionally with
# hyphens or digits inside (covers "state-of-the-art", "covid-19"), OR
# a pure digit run.
TOKEN_RE = re.compile(
    r"[A-Za-z\u00C0-\u024F][A-Za-z\u00C0-\u024F0-9\-]*"
    r"|[0-9]+"
)


def clean_text(raw: str) -> list[str]:
    """Return a list of cleaned sentence strings (one sentence per item).
    Each sentence is lowercased, space-separated tokens, no punctuation."""
    # Collapse all whitespace runs so paragraph breaks don't split
    # sentences mid-stream.
    raw = re.sub(r"\s+", " ", raw).strip()
    if not raw:
        return []
    out = []
    for sent in SENT_SPLIT_RE.split(raw):
        toks = TOKEN_RE.findall(sent)
        if not toks:
            continue
        out.append(" ".join(t.lower() for t in toks))
    return out


# Quick sanity check.
sample = "It was the best of times. It was the worst of times — Dickens said so!"
for line in clean_text(sample):
    print(repr(line))

'it was the best of times'
'it was the worst of times dickens said so'


In [4]:
from tqdm.auto import tqdm

NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(INPUT_DIR.glob("*.txt"))
if not input_files:
    raise FileNotFoundError(
        f"No .txt files found in {INPUT_DIR}. Check the path."
    )

print(f"Found {len(input_files)} .txt files in {INPUT_DIR}")
print()

total_chars_in = 0
total_sentences_out = 0
total_tokens_out = 0

for src in tqdm(input_files, unit="file", desc="cleaning"):
    with open(src, "r", encoding="utf-8", errors="replace") as f:
        raw = f.read()
    total_chars_in += len(raw)

    sentences = clean_text(raw)

    dst = NORMALIZED_DIR / src.name   # keep the original filename
    with open(dst, "w", encoding="utf-8") as f:
        for line in sentences:
            f.write(line)
            f.write("\n")

    total_sentences_out += len(sentences)
    total_tokens_out += sum(line.count(" ") + 1 for line in sentences)

print()
print(f"Input chars:      {total_chars_in:>12,}")
print(f"Output sentences: {total_sentences_out:>12,}")
print(f"Output tokens:    {total_tokens_out:>12,}")
print(f"Written to:       {NORMALIZED_DIR}/")

/Users/jacek_bakowski/DHSI_2026/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 27 .txt files in /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus



cleaning: 100%|██████████| 27/27 [00:02<00:00, 10.44file/s]


Input chars:        33,853,792
Output sentences:      271,211
Output tokens:       6,448,175
Written to:       /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus_normalized/


## Stage 2 — train

`PathLineSentences` reads every file in `NORMALIZED_DIR` and treats each
line as one sentence. A small callback gives us a per-epoch progress bar.

In [5]:
import time
import logging
from gensim.models import Word2Vec
from gensim.models.word2vec import PathLineSentences
from gensim.models.callbacks import CallbackAny2Vec

# Quiet down gensim's INFO logger so the progress bar is readable.
logging.getLogger("gensim").setLevel(logging.WARNING)


class EpochProgress(CallbackAny2Vec):
    def __init__(self, total):
        self.total = total
        self.bar = None

    def on_train_begin(self, model):
        self.bar = tqdm(total=self.total, unit="epoch", desc="training")

    def on_epoch_end(self, model):
        if self.bar:
            self.bar.update(1)

    def on_train_end(self, model):
        if self.bar:
            self.bar.close()


print("Word2Vec parameters:")
for k, v in W2V_PARAMS.items():
    print(f"  {k:<13} {v}")
print()

sentences = PathLineSentences(str(NORMALIZED_DIR))
callbacks = [EpochProgress(W2V_PARAMS["epochs"])]

t0 = time.time()
model = Word2Vec(sentences=sentences, callbacks=callbacks, **W2V_PARAMS)
elapsed = time.time() - t0

print()
print(f"Trained in    {elapsed:.1f}s")
print(f"Vocab size:   {len(model.wv):,} words")
print(f"Vector dim:   {model.wv.vector_size}")

Word2Vec parameters:
  vector_size   100
  window        5
  min_count     5
  sg            0
  workers       4
  epochs        5
  seed          42



training: 100%|██████████| 5/5 [00:07<00:00,  1.60s/epoch]


Trained in    8.9s
Vocab size:   19,344 words
Vector dim:   100


## Stage 3 — save

The model file goes into `<INPUT_DIR>_model/<INPUT_DIR_name>.model`.
Load it later with `Word2Vec.load(path)`.

In [6]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / f"{INPUT_DIR.name}.model"
model.save(str(model_path))

print(f"Saved: {model_path}")
print(f"Size:  {model_path.stat().st_size / (1024*1024):.1f} MB")

Saved: /Users/jacek_bakowski/DHSI_2026/A_Small_Collection_of_British_Fiction/lemmatized_corpus_model/lemmatized_corpus.model
Size:  15.3 MB


## Sanity check

A quick look at what the model learned. We don't hardcode any specific
words — instead we pick a handful of mid-frequency words from the
vocabulary itself, so this cell works whatever language your corpus is
in. (Skipping the very-top tokens, which are almost always function
words / stopwords that aren't pedagogically interesting.)

If you want to probe specific words instead, replace `probes` below
with your own list.

In [7]:
# Auto-pick probes: take 5 mid-frequency words from the vocabulary, so
# this cell works whatever language your corpus is in. The very-top
# tokens (function words) and the rare tail are both uninteresting;
# we want something in between.
vocab = model.wv.index_to_key
if len(vocab) >= 100:
    # Large vocab: skip the first 50 (likely stopwords), take next 5.
    probes = vocab[50:55]
elif len(vocab) >= 10:
    # Small vocab: skip ~20% from top, take 5.
    start = max(1, len(vocab) // 5)
    probes = vocab[start:start + 5]
else:
    probes = vocab[:5]

# Or use your own:
# probes = ["love", "death", "house", "money"]

for word in probes:
    print(f"\n{word} →")
    for w, s in model.wv.most_similar(word, topn=8):
        print(f"  {w:<20} {s:.3f}")


see →
  know                 0.630
  meet                 0.604
  tell                 0.594
  behold               0.580
  remember             0.577
  recollect            0.575
  hear                 0.569
  desert               0.567

very →
  extremely            0.788
  so                   0.647
  too                  0.642
  exceedingly          0.580
  tolerably            0.571
  remarkably           0.548
  uncommonly           0.507
  enough               0.505

who →
  whom                 0.805
  whose                0.489
  young                0.442
  whereas              0.393
  poor                 0.389
  medical              0.383
  which                0.380
  reverend             0.365

there →
  therebe              0.521
  obstacle             0.421
  doubt                0.410
  objection            0.403
  else                 0.376
  misunderstanding     0.376
  visible              0.373
  difference           0.368

come →
  go                   0.732
  br